# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Display basic metadata information
metadata_json = dataset.metadata.to_json()
print(f"Dataset Name: {getattr(dataset.metadata, 'name', '')}")
print(f"Description:  {getattr(dataset.metadata, 'description', '')}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', '')}")
print(f"Version:   {getattr(dataset.metadata, 'version', '')}")
print(f"License:   {getattr(dataset.metadata, 'license', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for exploration.

Below, we enumerate all available record sets, and for each record set, show the available fields and columns. All are referenced by their `@id`s as per Croissant/RDF best practice.

In [ ]:
# Let's enumerate all record set @ids, field @ids, and column @ids

def get_metadata_RDFS(ds):
    """Get record sets and their fields from metadata."""
    record_sets = getattr(ds.metadata, 'recordSet', None)
    if record_sets is None:
        print("No record sets found in the dataset metadata.")
        return
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    for rs in record_sets:
        rs_id = getattr(rs, '@id', getattr(rs, 'id', None))
        rs_name = getattr(rs, 'name', None)
        print(f"\nRecordSet: {rs_id} ({rs_name})")
        fields = getattr(rs, 'field', [])
        if not fields:
            print("  No fields listed.")
        else:
            for f in fields:
                f_id = getattr(f, '@id', getattr(f, 'id', None))
                f_name = getattr(f, 'name', None)
                f_dtype = getattr(f, 'dataType', None)
                print(f"  Field: {f_id}, name={f_name}, type={f_dtype}")
                cols = getattr(f, 'column', [])
                if cols:
                    if not isinstance(cols, list):
                        cols = [cols]
                    for col in cols:
                        col_id = getattr(col, '@id', getattr(col, 'id', None))
                        col_name = getattr(col, 'name', None)
                        print(f"    Column: {col_id}, name={col_name}")

get_metadata_RDFS(dataset)

If no record sets are displayed above, or the output is empty, let's list the available record sets, fields, and columns directly from the `.records()` interface.

In [ ]:
# If direct field introspection failed above, try listing all available record sets via .records()
# This will only print the first N items per record set for sampling.
record_set_ids = []
try:
    record_set_ids = dataset.record_set_ids
except Exception:
    # Fallback: try known/suspected main record set id
    record_set_ids = ["https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034"]

print("Available Record Set `@id`s:")
for rsid in record_set_ids:
    print("  -", rsid)

# Display a sample of records for each record set
for rsid in record_set_ids:
    print(f"\nSample records for record set {rsid}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=rsid)):
            print(json.dumps(rec, indent=2) if i == 0 else rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"  Could not iterate records: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Define the record set(s) you want to extract. Insert correct `@id`s from overview cell(s).
# You can replace this list with the output from the previous cell.
record_sets = [
    "https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034"
]
dataframes = {}

for record_set_id in record_sets:
    print(f"\nExtracting records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"  Loaded {df.shape[0]} rows, {df.shape[1]} columns.")
    print("  Columns:", df.columns.tolist())
    dataframes[record_set_id] = df

# Select main record set for demonstration below
main_record_set_id = "https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034"
print("\nFirst few records:")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

The following example assumes there's a field representing a numeric measurement (e.g., "MW_kDa", (molecular weight in kDa)). Adjust the field name to match the dataset's columns.

In [ ]:
# Choose a numeric field for demonstration
# Try 'MW_kDa' (Molecular Weight), falling back to another numeric column if needed.
df = dataframes[main_record_set_id]

# Discover potential numeric columns
numeric_columns = df.select_dtypes(include='number').columns.tolist()
print(f"Numeric columns in the dataset: {numeric_columns}")

if len(numeric_columns) == 0:
    print("No numeric columns found! Check column names and types above and try again.")
else:
    numeric_field_id = numeric_columns[0]
    print(f"Using numeric field: " + numeric_field_id)
    # Filter for non-null and above threshold
    threshold = df[numeric_field_id].quantile(0.75) if df[numeric_field_id].notnull().sum() else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by a potential categorical variable, e.g., 'Sample' or 'SampleGroup', if it exists
    possible_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col.lower().startswith('sample')]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field} (mean):")
        print(grouped_df.head())
    else:
        print("\nNo suitable group field detected for grouping analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the numeric field and, if available, compare means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_columns) > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if a group field was found
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"Distribution of {numeric_field_id} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² human extracellular vesicle protein dataset, explored its schema, loaded a primary record set using the `mlcroissant` library, and performed basic exploratory data analysis. We investigated the distribution of one of the key numeric fields and grouped data by available attributes as appropriate.

To go further, consider combining this data with other Croissant-linked resources, training models, or applying advanced visualizations.

----
*Notebook generated using the mlcroissant and FAIR² standards. Cite as: Kulka, M., Rodriguez Miera, S., Marcet-Palacios, M. (2026). Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells.*